# Title: Data Exploration
### Author: Kolbe Sussman

# Load dependancies and data


In [19]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast



In [20]:
df = pd.read_csv("../data/processed/umich_works_cleaned.csv")
df.head()

,title,id,doi,authorships,topics,primary_topic,cited_by_count,publication_year,related_works,concepts,authorships_parsed,author_ids,author_names,institutions,raw_affiliations,topics_parsed,topic_ids,display_names,topic_scores
0,"""A Young Girls Dream"": Examining the Barriers ...",https://openalex.org/W2059047491,https://doi.org/10.5429/2079-3871(2010)v1i1.1en,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T11113', 'displa...","{'id': 'https://openalex.org/T11113', 'display...",42,2011,"['https://openalex.org/W4391375266', 'https://...","[{'id': 'https://openalex.org/C109747225', 'wi...","[{'author_position': 'first', 'author': {'id':...",['https://openalex.org/A5018521540'],['Monique Bourdage'],"['University of Michigan–Ann Arbor', 'Girls In...","['Girls Rock Denver', '(University of Michigan)']","[{'id': 'https://openalex.org/T11113', 'displa...","['https://openalex.org/T11113', 'https://opena...","['Music History and Culture', 'Diverse Music E...","[0.9998000264167786, 0.9939000010490417, 0.969..."
1,"""Anatometabolic"" tumor imaging: fusion of FDG ...",https://openalex.org/W2154563058,NaN,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T10522', 'displa...","{'id': 'https://openalex.org/T10522', 'display...",220,1993,"['https://openalex.org/W2012261854', 'https://...","[{'id': 'https://openalex.org/C173974348', 'wi...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/A5024722162', 'https://...","['Richard L. Wahl', 'Leslie E. Quint', 'Richar...",['University of Michigan–Ann Arbor'],"['University of Michigan Ann Arbor', 'Departme...","[{'id': 'https://openalex.org/T10522', 'displa...","['https://openalex.org/T10522', 'https://opena...",['Medical Imaging Techniques and Applications'...,"[0.9995999932289124, 0.9975000023841858, 0.996..."
2,"""At the End of the Day Facebook Does What ItWa...",https://openalex.org/W3094140017,https://doi.org/10.1145/3415238,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T12262', 'displa...","{'id': 'https://openalex.org/T12262', 'display...",69,2020,"['https://openalex.org/W2735088306', 'https://...","[{'id': 'https://openalex.org/C2777582232', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/A5064496195', 'https://...","['Kristen Vaccaro', 'Christian Sandvig', 'Karr...","['University of Illinois Urbana-Champaign', 'U...","['University of Michigan', 'University of Illi...","[{'id': 'https://openalex.org/T12262', 'displa...","['https://openalex.org/T12262', 'https://opena...","['Hate Speech and Cyberbullying Detection', 'E...","[0.9997000098228455, 0.9990000128746033, 0.985..."
3,"""Body-In-The-Loop"": Optimizing Device Paramete...",https://openalex.org/W1864504421,https://doi.org/10.1371/journal.pone.0135342,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T11023', 'displa...","{'id': 'https://openalex.org/T11023', 'display...",142,2015,"['https://openalex.org/W2091389660', 'https://...","[{'id': 'https://openalex.org/C186633575', 'wi...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/A5065760489', 'https://...","['Wyatt Felt', 'Jessica C. Selinger', 'J. Maxw...","['University of Michigan–Ann Arbor', 'Simon Fr...",['Department of Biomedical Physiology and Kine...,"[{'id': 'https://openalex.org/T11023', 'displa...","['https://openalex.org/T11023', 'https://opena...","['Prosthetics and Rehabilitation Robotics', 'M...","[0.9998000264167786, 0.9934999942779541, 0.987..."
4,"""Culture of Drinking"" and Individual Problems ...",https://openalex.org/W2111177400,https://doi.org/10.1093/aje/kwn022,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T10043', 'displa...","{'id': 'https://openalex.org/T10043', 'display...",132,2008,"['https://openalex.org/W2118512822', 'https://...","[{'id': 'https://openalex.org/C2776679223', 'w...","[{'author_posit

## Authorship networks

In [21]:
df = df.groupby('title').first().reset_index()   # delete later - this was added to the preprocessing.py


In [22]:
def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

df['author_names'] = df['author_names'].apply(safe_parse)
df['raw_affiliations'] = df['raw_affiliations'].apply(safe_parse)
df['author_ids'] = df['author_ids'].apply(safe_parse)


In [23]:
# Extract author-level edges using pre-parsed column
MAX_AUTHORS = 20  # skip papers with too many authors

edge_weights = Counter()
author_info = {}

for row in df.itertuples(index=False):
    author_names = row.author_names
    raw_affiliations = row.raw_affiliations
    author_ids = row.author_ids

    if not author_names or len(author_names) < 2:
        continue

    if len(author_names) > MAX_AUTHORS:
        continue

    # remove duplicates just in case
    author_names = list(set(author_names))

    # store metadata
    for i, name in enumerate(author_names):
        if name not in author_info:
            author_info[name] = {
                'id': author_ids[i] if i < len(author_ids) else None,
                'affiliation': raw_affiliations[i] if i < len(raw_affiliations) else None
            }

    # generate edges
    for a, b in combinations(sorted(author_names), 2):
        edge_weights[(a, b)] += 1

In [24]:
total = len(edge_weights)

print(total)

1498395


In [25]:
# Build NetworkX graph
G = nx.Graph()

for i, ((auth1, auth2), weight) in enumerate(edge_weights.items(), start=1):
    G.add_edge(auth1, auth2, weight=weight)
    
    if i % 10000 == 0:
        print(f"Processed {i}/{total} edges ({i/total:.1%})")



Processed 10000/1498395 edges (0.7%)
Processed 20000/1498395 edges (1.3%)
Processed 30000/1498395 edges (2.0%)
Processed 40000/1498395 edges (2.7%)
Processed 50000/1498395 edges (3.3%)
Processed 60000/1498395 edges (4.0%)
Processed 70000/1498395 edges (4.7%)
Processed 80000/1498395 edges (5.3%)
Processed 90000/1498395 edges (6.0%)
Processed 100000/1498395 edges (6.7%)
Processed 110000/1498395 edges (7.3%)
Processed 120000/1498395 edges (8.0%)
Processed 130000/1498395 edges (8.7%)
Processed 140000/1498395 edges (9.3%)
Processed 150000/1498395 edges (10.0%)
Processed 160000/1498395 edges (10.7%)
Processed 170000/1498395 edges (11.3%)
Processed 180000/1498395 edges (12.0%)
Processed 190000/1498395 edges (12.7%)
Processed 200000/1498395 edges (13.3%)
Processed 210000/1498395 edges (14.0%)
Processed 220000/1498395 edges (14.7%)
Processed 230000/1498395 edges (15.3%)
Processed 240000/1498395 edges (16.0%)
Processed 250000/1498395 edges (16.7%)
Processed 260000/1498395 edges (17.4%)
Processed

In [26]:
# Compute metrics
degree_centrality = nx.degree_centrality(G)
print("done")
#betweenness_centrality = nx.betweenness_centrality(G, weight='weight')
eigenvector_centrality = nx.eigenvector_centrality(G, weight='weight')
print("done")


done
done


In [27]:

# Save network metrics
metrics_df = pd.DataFrame({
    'author': list(G.nodes),
    'degree_centrality': [degree_centrality[a] for a in G.nodes],
    #'betweenness_centrality': [betweenness_centrality[a] for a in G.nodes],
    'eigenvector_centrality': [eigenvector_centrality[a] for a in G.nodes],
    'id': [author_info[a]['id'] for a in G.nodes],
    'affiliation': [author_info[a]['affiliation'] for a in G.nodes]
})

#metrics_df.to_csv("../data/processed/author_network_metrics.csv", index=False)

In [28]:
# Save full edge list
edges_df = pd.DataFrame([
    {'author_1': a1, 'author_2': a2, 'weight': w}
    for (a1, a2), w in edge_weights.items()
])
#edges_df.to_csv("../data/processed/author_network_edges.csv", index=False)

#print("Author network created and metrics saved!")

In [29]:
# Graph top XX collaborations?

## Department Networks

In [30]:
def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []


# Parse affiliations
df['raw_affiliations'] = df['raw_affiliations'].apply(safe_parse)

MAX_AFFILIATIONS = 20  # same idea as authors

edge_weights = Counter()

for row in df.itertuples(index=False):
    affiliations = row.raw_affiliations

    if not affiliations or len(affiliations) < 2:
        continue

    if len(affiliations) > MAX_AFFILIATIONS:
        continue

    # 🔑 IMPORTANT: deduplicate within paper
    affiliations = list(set([a for a in affiliations if a]))

    if len(affiliations) < 2:
        continue

    # generate edges
    for a, b in combinations(sorted(affiliations), 2):
        edge_weights[(a, b)] += 1

# Build graph (FIXED INDENTATION BUG)
G = nx.Graph()

for (aff1, aff2), weight in edge_weights.items():
    G.add_edge(aff1, aff2, weight=weight)


In [31]:
# Compute metrics
degree_centrality = nx.degree_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G, weight='weight')

# Save metrics
metrics_df = pd.DataFrame({
    'affiliation': list(G.nodes),
    'degree_centrality': [degree_centrality[a] for a in G.nodes],
    'eigenvector_centrality': [eigenvector_centrality[a] for a in G.nodes],
})

#metrics_df.to_csv("data/processed/affiliation_network_metrics.csv", index=False)

# Save edges
edges_df = pd.DataFrame([
    {'affiliation_1': a1, 'affiliation_2': a2, 'weight': w}
    for (a1, a2), w in edge_weights.items()
])

#edges_df.to_csv("data/processed/affiliation_network_edges.csv", index=False)

In [32]:
# create figures

## Topic Networks

In [33]:
def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []


# Parse topics
df['display_names'] = df['display_names'].apply(safe_parse)

MAX_TOPICS = 50  # topics are usually fewer → can allow more

edge_weights = Counter()

for row in df.itertuples(index=False):
    topics = row.display_names

    if not topics or len(topics) < 2:
        continue

    if len(topics) > MAX_TOPICS:
        continue

    # 🔑 Deduplicate topics within a paper
    topics = list(set([t for t in topics if t]))

    if len(topics) < 2:
        continue

    # generate edges
    for t1, t2 in combinations(sorted(topics), 2):
        edge_weights[(t1, t2)] += 1

# Build graph
G = nx.Graph()

for (t1, t2), weight in edge_weights.items():
    G.add_edge(t1, t2, weight=weight)

In [34]:
def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []


# Parse topics
df['display_names'] = df['display_names'].apply(safe_parse)

MAX_TOPICS = 50  # topics are usually fewer → can allow more

edge_weights = Counter()

for row in df.itertuples(index=False):
    topics = row.display_names

    if not topics or len(topics) < 2:
        continue

    if len(topics) > MAX_TOPICS:
        continue

    # 🔑 Deduplicate topics within a paper
    topics = list(set([t for t in topics if t]))

    if len(topics) < 2:
        continue

    # generate edges
    for t1, t2 in combinations(sorted(topics), 2):
        edge_weights[(t1, t2)] += 1

# Build graph
G = nx.Graph()

for (t1, t2), weight in edge_weights.items():
    G.add_edge(t1, t2, weight=weight)


In [35]:
# Compute metrics
degree_centrality = nx.degree_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G, weight='weight')

# Save metrics
metrics_df = pd.DataFrame({
    'topic': list(G.nodes),
    'degree_centrality': [degree_centrality[t] for t in G.nodes],
    'eigenvector_centrality': [eigenvector_centrality[t] for t in G.nodes],
})

#metrics_df.to_csv("data/processed/topic_network_metrics.csv", index=False)

# Save edges
edges_df = pd.DataFrame([
    {'topic_1': t1, 'topic_2': t2, 'weight': w}
    for (t1, t2), w in edge_weights.items()
])

#edges_df.to_csv("data/processed/topic_network_edges.csv", index=False)

In [ ]:
#graph topic network